# 💰 LTV — Valor del Cliente (Ridge Regression)
> **Caso de negocio:** Crisol · Valor de Cliente / CRM
> **Framework:** CRISP-DM · Digital Marketing Analytics — UPC
> **Autor:** Miguel Salazar · Attach Group

---

## Contexto del problema

Crisol (librería online) gestiona miles de clientes registrados, pero el presupuesto de marketing (emailing, descuentos, campañas de reactivación) se distribuye **igual para todos los clientes**, sin considerar cuánto vale cada uno a futuro.

**Objetivo:** Predecir el LTV anual (S/.) de cada cliente a partir de su comportamiento de compra, para asignar presupuesto de marketing y CRM de forma diferenciada.

**KPIs de éxito:**
- R² >= 0.55 en test set
- MAPE <= 30% (error porcentual aceptable para presupuestos)
- Identificar el top 20% de clientes que concentra la mayor parte del LTV

**Algoritmo:** Ridge Regression (regresión lineal con regularización L2, coeficientes directamente interpretables)

## 📦 Fase 0 — Setup

In [ ]:
!pip install -q plotly

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Setup completo ✓')

## 1️⃣ Fase 1 — Business Understanding

In [ ]:
PROBLEMA = {
    'tipo': 'Regresión supervisada',
    'target': 'ltv_anual (S/. proyectados para los próximos 12 meses)',
    'ventana_prediccion': '12 meses, a partir del comportamiento de los últimos 6 meses',
    'features': [
        'compras_6m    → número de compras en los últimos 6 meses',
        'ticket_prom   → ticket promedio de compra (S/.)',
        'meses_cliente → antigüedad como cliente (meses)',
        'categorias    → número de categorías de producto distintas compradas',
        'canal_digital → 1 si el cliente compra principalmente por canal digital, 0 si es tienda física',
    ],
    'criterios_aceptacion': {
        'R2':   '>= 0.55',
        'MAPE': '<= 30%',
    }
}

for k, v in PROBLEMA.items():
    if isinstance(v, list):
        print(f'\n{k}:')
        for item in v: print(f'  {item}')
    elif isinstance(v, dict):
        print(f'\n{k}:')
        for mk, mv in v.items(): print(f'  {mk}: {mv}')
    else:
        print(f'{k}: {v}')

## 2️⃣ Fase 2 — Data Understanding

In [ ]:
# Generación de datos sintéticos con estructura realista
# En producción: reemplazar con datos del Customer Data Platform / BigQuery
N = 1200

compras_6m    = np.random.randint(1, 21, N)
ticket_prom   = (np.round(np.random.uniform(50, 800, N) / 10) * 10).astype(int)
meses_cliente = np.random.randint(1, 61, N)
categorias    = np.random.randint(1, 8, N)
edad          = np.random.randint(18, 68, N)
canal_digital = (np.random.random(N) > 0.4).astype(int)

# DGP: el LTV crece con frecuencia, ticket, antigüedad, variedad de categorías y canal digital
ltv_anual = np.round(
    180 * compras_6m
    + 3.2 * ticket_prom
    + 28 * meses_cliente
    + 120 * categorias
    + 200 * canal_digital
    - 200
    + np.random.normal(0, 300, N)
)
ltv_anual = np.maximum(ltv_anual, 100).astype(int)

df = pd.DataFrame({
    'cliente_id':    range(1, N+1),
    'compras_6m':    compras_6m,
    'ticket_prom':   ticket_prom,
    'meses_cliente': meses_cliente,
    'categorias':    categorias,
    'edad':          edad,
    'canal_digital': canal_digital,
    'ltv_anual':     ltv_anual
})

print(f'Dataset: {df.shape}')
print(f'\nLTV anual — estadísticas:')
print(df['ltv_anual'].describe().round(1))
df.head()

In [ ]:
# Distribución del target y de las features numéricas
features = ['compras_6m', 'ticket_prom', 'meses_cliente', 'categorias']

fig = make_subplots(rows=1, cols=2, subplot_titles=['Distribución de LTV anual', 'LTV por canal'])

fig.add_trace(go.Histogram(x=df['ltv_anual'], marker_color='#1a4c8c', nbinsx=30,
                            showlegend=False), row=1, col=1)

for canal, label, color in [(0, 'Tienda física', '#c0392b'), (1, 'Digital', '#0d9488')]:
    fig.add_trace(go.Box(y=df[df['canal_digital']==canal]['ltv_anual'],
                          name=label, marker_color=color, showlegend=False),
                  row=1, col=2)

fig.update_layout(height=400, title='LTV anual — distribución general y por canal',
                  plot_bgcolor='white', paper_bgcolor='white')
fig.show()

In [ ]:
# Matriz de correlación entre features y el target
num_cols = ['compras_6m', 'ticket_prom', 'meses_cliente', 'categorias', 'canal_digital', 'ltv_anual']
corr = df[num_cols].corr().round(2)

fig = px.imshow(corr, text_auto=True, color_continuous_scale='RdBu_r',
                zmin=-1, zmax=1, title='Correlación entre variables y LTV anual')
fig.update_layout(height=450, paper_bgcolor='white')
fig.show()

In [ ]:
# Relación entre compras_6m y ltv_anual, coloreado por canal
fig = px.scatter(df, x='compras_6m', y='ltv_anual', color='canal_digital',
                 color_continuous_scale=['#c0392b', '#0d9488'],
                 title='LTV anual vs. frecuencia de compra (6 meses)',
                 labels={'compras_6m': 'Compras (6 meses)', 'ltv_anual': 'LTV anual (S/.)'},
                 opacity=0.6)
fig.update_layout(height=420, plot_bgcolor='white', paper_bgcolor='white')
fig.show()

## 3️⃣ Fase 3 — Data Preparation

In [ ]:
FEATURES = ['compras_6m', 'ticket_prom', 'meses_cliente', 'categorias', 'canal_digital']
TARGET   = 'ltv_anual'

X = df[FEATURES].fillna(0)
y = df[TARGET].astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'LTV promedio train: S/. {y_train.mean():,.0f}')
print(f'LTV promedio test:  S/. {y_test.mean():,.0f}')

## 4️⃣ Fase 4 — Modeling

In [ ]:
# Ridge Regression: regularización L2 reduce overfitting y mantiene coeficientes interpretables
model = Ridge(alpha=1.0)
model.fit(X_train, y_train)

# Validación cruzada
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_r2 = cross_val_score(model, X_train, y_train, cv=cv, scoring='r2')

print(f'CV R²: {cv_r2.mean():.3f} ± {cv_r2.std():.3f}')
print(f'Intercepto: {model.intercept_:,.1f}')

In [ ]:
y_pred = model.predict(X_test)

# Coeficientes del modelo
coef_df = pd.DataFrame({
    'feature': FEATURES,
    'coef': model.coef_
}).sort_values('coef', ascending=True)

fig = go.Figure(go.Bar(
    x=coef_df['coef'], y=coef_df['feature'], orientation='h',
    marker_color=['#0d9488' if c > 0 else '#c0392b' for c in coef_df['coef']],
    text=coef_df['coef'].map('{:.1f}'.format), textposition='outside'
))
fig.update_layout(title='Coeficientes del modelo Ridge (impacto en S/. de LTV anual por unidad)',
                  height=350, plot_bgcolor='white', paper_bgcolor='white',
                  xaxis=dict(gridcolor='#f0f0f0', zeroline=True, zerolinecolor='#aaa'),
                  yaxis=dict(showgrid=False))
fig.show()
display(coef_df.sort_values('coef', ascending=False))

## 5️⃣ Fase 5 — Evaluation

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / (y_test + 1e-9))) * 100
r2   = r2_score(y_test, y_pred)

metrics = {'R2': r2, 'RMSE': rmse, 'MAE': mae, 'MAPE (%)': mape}
criterios = {'R2': 0.55, 'MAPE (%)': 30}

print('=== MÉTRICAS EN TEST SET ===')
for k, v in metrics.items():
    umbral = criterios.get(k)
    estado = ''
    if umbral is not None:
        if k == 'MAPE (%)':
            estado = '✅ APROBADO' if v <= umbral else f'❌ NECESITA MEJORA (umbral <= {umbral})'
        else:
            estado = '✅ APROBADO' if v >= umbral else f'❌ NECESITA MEJORA (umbral >= {umbral})'
    print(f'  {k:10s}: {v:,.2f}  {estado}')

In [ ]:
# Predicho vs Real
fig = go.Figure()
fig.add_trace(go.Scatter(x=y_test, y=y_pred, mode='markers',
                          marker=dict(color='#1a4c8c', opacity=0.5, size=6),
                          name='Predicción'))
mn, mx = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
fig.add_trace(go.Scatter(x=[mn, mx], y=[mn, mx], mode='lines',
                          line=dict(color='#c0392b', width=1, dash='dash'),
                          name='Ideal (y=x)'))
fig.update_layout(title='LTV Predicho vs Real (test set)',
                  xaxis_title='LTV real (S/.)', yaxis_title='LTV predicho (S/.)',
                  height=420, plot_bgcolor='white', paper_bgcolor='white')
fig.update_xaxes(gridcolor='#f0f0f0')
fig.update_yaxes(gridcolor='#f0f0f0')
fig.show()

In [ ]:
# Análisis de residuos
residuos = y_test.values - y_pred

fig = make_subplots(rows=1, cols=2, subplot_titles=['Residuos vs Predicho', 'Distribución de residuos'])

fig.add_trace(go.Scatter(x=y_pred, y=residuos, mode='markers',
                          marker=dict(color='#7c3aed', opacity=0.5, size=6),
                          showlegend=False), row=1, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='gray', row=1, col=1)

fig.add_trace(go.Histogram(x=residuos, marker_color='#7c3aed', nbinsx=30,
                            showlegend=False), row=1, col=2)

fig.update_layout(height=380, title='Diagnóstico de residuos',
                  plot_bgcolor='white', paper_bgcolor='white')
fig.update_xaxes(title_text='LTV predicho (S/.)', row=1, col=1)
fig.update_yaxes(title_text='Residuo (S/.)', row=1, col=1)
fig.show()

## 6️⃣ Fase 6 — Deployment

In [ ]:
# Score de LTV para toda la base + segmentación por valor
df['ltv_predicho'] = model.predict(df[FEATURES].fillna(0)).round(0)

df['segmento_valor'] = pd.qcut(
    df['ltv_predicho'], q=4,
    labels=['Bronce', 'Plata', 'Oro', 'Platino']
)

resumen_seg = df.groupby('segmento_valor').agg(
    n_clientes=('cliente_id', 'count'),
    ltv_promedio=('ltv_predicho', 'mean'),
    ltv_total=('ltv_predicho', 'sum')
).round(1)
resumen_seg['pct_ltv_total'] = (resumen_seg['ltv_total'] / resumen_seg['ltv_total'].sum() * 100).round(1)

print('Segmentación de clientes por LTV proyectado:')
display(resumen_seg)

fig = px.bar(resumen_seg.reset_index(), x='segmento_valor', y='ltv_total',
             title='LTV total proyectado por segmento de valor',
             text=resumen_seg['pct_ltv_total'].map('{:.0f}%'.format),
             color='segmento_valor',
             color_discrete_sequence=['#c0392b', '#d97706', '#0d9488', '#1a4c8c'])
fig.update_traces(textposition='outside')
fig.update_layout(height=380, plot_bgcolor='white', paper_bgcolor='white', showlegend=False)
fig.show()

In [ ]:
# Guardar outputs
df[['cliente_id', 'ltv_predicho', 'segmento_valor']].to_csv('clientes_ltv_crisol.csv', index=False)
print('Archivo clientes_ltv_crisol.csv guardado ✓')

import joblib
joblib.dump({'model': model, 'features': FEATURES}, 'modelo_ltv_crisol.pkl')
print('Modelo modelo_ltv_crisol.pkl guardado ✓')

In [ ]:
# Resumen ejecutivo
n_clientes   = len(df)
ltv_total    = df['ltv_predicho'].sum()
platino      = resumen_seg.loc['Platino']
pct_platino_clientes = platino['n_clientes'] / n_clientes * 100

print('=== RESUMEN EJECUTIVO ===')
print(f'Clientes analizados:                {n_clientes:,}')
print(f'LTV total proyectado (12 meses):    S/. {ltv_total:,.0f}')
print(f'Segmento Platino:                   {platino["n_clientes"]:,.0f} clientes ({pct_platino_clientes:.0f}%)')
print(f'                                     concentra {platino["pct_ltv_total"]:.0f}% del LTV total')
print(f'\nArquitectura de producción:')
print('  CRM + transacciones → BigQuery → score mensual de LTV → segmentación CRM (emailing, beneficios)')